# SmartBasket · Notebook 03 — PCY vs A-Priori on all 3.2 million orders

**Team Frenchies** · Introduction to Big Data · Week 11

**Why this notebook.** Notebook 02 showed that A-Priori runs on the full dataset at a 0.05 % support, but then only **763 products** can ever be recommended (1.5 % of the catalogue). To cover more products we must **lower the support**. Lower support means more frequent items, and pass 2 of A-Priori has to count every pair of them: memory explodes.

**PCY** (MMDS §6.3.1) uses the memory that is idle during pass 1: every pair of every basket is hashed into a bucket and the bucket count goes up. In pass 2, a pair is counted only if **both items are frequent AND its bucket is frequent**.

How we compare the two, with the textbook's memory model (MMDS §6.2.1 and §6.3.1):

| Method | What pass 2 holds in memory | Bytes |
|---|---|---|
| A-Priori, triangular matrix | one counter for every possible pair of frequent items, C(F, 2) | 4 per counter |
| A-Priori, table of triples | one entry per pair that actually occurs | 12 per pair |
| PCY | one entry per candidate pair (both items frequent and bucket frequent) + a bitmap | 12 per pair + B/8 |

PCY pass 1 also holds the bucket array: 4 bytes × B buckets.

> 🇫🇷 *On ne mesure pas que le temps : comme au lab 6, on compte combien de compteurs chaque méthode doit garder en mémoire au passage 2. On vérifie aussi que PCY trouve exactement les mêmes paires fréquentes qu'A-Priori.*

In [ ]:
# 🇫🇷 Imports et paramètres
import os, time, json, math, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
OUT = "out/pcy_week11"
os.makedirs(OUT, exist_ok=True)

SUPPORTS_PCT = [0.05, 0.02, 0.01, 0.005]       # % of baskets; 0.05 % is the threshold kept in notebook 02
MAIN_BUCKETS = 2**24                           # 16.8 M buckets = 67 MB for the bucket array
BUCKET_SWEEP = [2**20, 2**22, 2**24]           # tested at BUCKET_SWEEP_SUPPORT only
BUCKET_SWEEP_SUPPORT = 0.01
MAX_TRIANGLE = 400_000_000                     # skip a support if C(F,2) is larger (would need > 1.6 GB per array)

In [ ]:
# 🇫🇷 Données : d'abord Google Drive (dossier « instacart »), sinon Kaggle
DRIVE_DIR = "/content/drive/MyDrive/instacart"
DATA_DIR = os.environ.get("SB_DATA_DIR")          # local test only

if DATA_DIR is None:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.exists(os.path.join(DRIVE_DIR, "order_products__prior.csv")):
            DATA_DIR = DRIVE_DIR
            print("Data found on Google Drive")
        else:
            print("No 'instacart' folder with the CSV files on Drive, trying Kaggle")
    except Exception as e:
        print("Google Drive not available:", type(e).__name__)

if DATA_DIR is None:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")

print("DATA_DIR =", DATA_DIR)

## 1 · Baskets as flat arrays

All 32 million lines are kept as two NumPy arrays: the products, sorted by order, and the position where each order starts. A basket is a slice of the product array. This holds all 3.2 M baskets in a few hundred MB, and lets us generate the pairs of thousands of baskets at once.

> 🇫🇷 *Pas de liste Python de 3,2 millions de paniers : deux tableaux NumPy suffisent, c'est ce qui rend le calcul possible dans Colab.*

In [ ]:
# 🇫🇷 Chargement : chaque commande devient une tranche d'un grand tableau de produits
t0 = time.time()
prior = pd.read_csv(os.path.join(DATA_DIR, "order_products__prior.csv"),
                    usecols=["order_id", "product_id"],
                    dtype={"order_id": "int32", "product_id": "int32"}).drop_duplicates()
order_ids = prior["order_id"].to_numpy()
products = prior["product_id"].to_numpy()
idx = np.lexsort((products, order_ids))         # sort by order, then by product inside the order
order_ids, items = order_ids[idx], products[idx]
del prior, products, idx; gc.collect()

starts = np.flatnonzero(np.r_[True, order_ids[1:] != order_ids[:-1]])
offsets = np.r_[starts, len(items)].astype(np.int64)
sizes = np.diff(offsets)
N = len(sizes)
P = int(items.max()) + 1
basket_of_line = np.repeat(np.arange(N, dtype=np.int32), sizes)
del order_ids, starts; gc.collect()

ALL_PAIRS = int((sizes * (sizes - 1) // 2).sum())
NAMES = pd.read_csv(os.path.join(DATA_DIR, "products.csv")).set_index("product_id")["product_name"].to_dict()
print(f"baskets: {N:,}   lines: {len(items):,}   distinct products: {len(np.unique(items)):,}")
print(f"pairs inside baskets (what PCY hashes in pass 1): {ALL_PAIRS:,}")
print(f"loaded in {time.time() - t0:.1f} s")

In [ ]:
# 🇫🇷 Générateur de paires : traite ensemble tous les paniers de même taille (rapide en NumPy)
_TRIU = {}

def iter_pairs(items, offsets, chunk_pairs=20_000_000):
    """Yield (a, b) arrays with every pair a < b inside each basket."""
    sizes = np.diff(offsets)
    for n in np.unique(sizes):
        if n < 2:
            continue
        if n not in _TRIU:
            _TRIU[n] = np.triu_indices(n, k=1)
        iu, ju = _TRIU[n]
        baskets = np.flatnonzero(sizes == n)
        step = max(1, chunk_pairs // len(iu))
        for s in range(0, len(baskets), step):
            first = offsets[baskets[s:s + step]]
            mat = items[first[:, None] + np.arange(n)]
            yield mat[:, iu].ravel(), mat[:, ju].ravel()

def bucket_of(a, b, n_buckets):
    """PCY hash of the pair (a, b) into one of n_buckets buckets."""
    key = a.astype(np.uint64) * np.uint64(P) + b.astype(np.uint64)
    return ((key * np.uint64(2654435761)) % np.uint64(2**32) % np.uint64(n_buckets)).astype(np.int64)

## 2 · Pass 1: count items (A-Priori) and hash every pair into buckets (PCY)

A-Priori pass 1 only counts items. PCY pass 1 also hashes all the pairs of every basket. We fill three bucket arrays of different sizes in the same pass, to see how the number of buckets changes the filter.

In [ ]:
# 🇫🇷 Passage 1 : comptage des produits (A-Priori) + comptage des paires par seau (PCY)
t0 = time.time()
item_counts = np.bincount(items, minlength=P)
t_pass1_apriori = time.time() - t0

bucket_counts = {B: np.zeros(B, dtype=np.int64) for B in BUCKET_SWEEP}
t_hash = {B: 0.0 for B in BUCKET_SWEEP}
t0 = time.time()
seen = 0
for a, b in iter_pairs(items, offsets):
    for B in BUCKET_SWEEP:
        t1 = time.perf_counter()
        bucket_counts[B] += np.bincount(bucket_of(a, b, B), minlength=B)
        t_hash[B] += time.perf_counter() - t1
    seen += len(a)
t_pass1_all = time.time() - t0
assert seen == ALL_PAIRS

print(f"A-Priori pass 1: {t_pass1_apriori:.2f} s")
for B in BUCKET_SWEEP:
    print(f"PCY pass 1 hashing into {B:>11,} buckets: {t_hash[B]:6.1f} s "
          f"(bucket array {4 * B / 1e6:6.1f} MB, mean {ALL_PAIRS / B:7.1f} pairs per bucket)")

## 3 · Pass 2 at each support

For each support we count, in one pass over the baskets reduced to their frequent items:
- every pair of frequent items (A-Priori), in a triangular array;
- only the pairs whose bucket is frequent (PCY candidates).

Then we check that both give exactly the same frequent pairs.

In [ ]:
# 🇫🇷 Passage 2 pour un seuil donné : A-Priori et PCY comptés en même temps, puis comparés
def tri_index(ia, ib, F):
    return ia * F - ia * (ia + 1) // 2 + (ib - ia - 1)

def tri_decode(k, F):
    i = (F - 2 - np.floor(np.sqrt(-8 * k + 4 * F * (F - 1) - 7) / 2.0 - 0.5)).astype(np.int64)
    j = (k + i + 1 - F * (F - 1) // 2 + (F - i) * ((F - i) - 1) // 2).astype(np.int64)
    return i, j

def run_support(support_pct, buckets):
    s = math.ceil(N * support_pct / 100)
    freq = item_counts >= s
    F = int(freq.sum())
    n_tri = F * (F - 1) // 2
    row = {"support_pct": support_pct, "support_baskets": s, "frequent_items": F,
           "apriori_triangular_counters": n_tri}
    if n_tri > MAX_TRIANGLE:
        print(f"{support_pct}%: C(F,2) = {n_tri:,} counters, skipped (over {MAX_TRIANGLE:,})")
        row["status"] = "skipped: triangular matrix too large"
        return row, None

    t0 = time.time()
    freq_ids = np.flatnonzero(freq)
    dense = np.full(P, -1, dtype=np.int64)
    dense[freq_ids] = np.arange(F)
    keep = freq[items]
    kept_items = items[keep]
    kept_offsets = np.r_[0, np.cumsum(np.bincount(basket_of_line[keep], minlength=N))].astype(np.int64)

    all_counts = np.zeros(n_tri, dtype=np.int32)
    bitmaps = {B: bucket_counts[B] >= s for B in buckets}
    cand_counts = {B: np.zeros(n_tri, dtype=np.int32) for B in buckets}
    for a, b in iter_pairs(kept_items, kept_offsets):
        tri = tri_index(dense[a], dense[b], F)
        u, c = np.unique(tri, return_counts=True)
        all_counts[u] += c.astype(np.int32)
        for B in buckets:
            in_bitmap = bitmaps[B][bucket_of(a, b, B)]
            u, c = np.unique(tri[in_bitmap], return_counts=True)
            cand_counts[B][u] += c.astype(np.int32)
    row["pass2_s"] = round(time.time() - t0, 1)

    frequent = np.flatnonzero(all_counts >= s)
    observed = int(np.count_nonzero(all_counts))
    row.update({"apriori_counted_pairs": observed, "frequent_pairs": int(len(frequent)),
                "apriori_triangular_MB": round(4 * n_tri / 1e6, 1),
                "apriori_triples_MB": round(12 * observed / 1e6, 1)})
    for B in buckets:
        cand = int(np.count_nonzero(cand_counts[B]))
        same = np.array_equal(frequent, np.flatnonzero(cand_counts[B] >= s))
        row[f"pcy_{B}_frequent_buckets_pct"] = round(100 * float(bitmaps[B].mean()), 2)
        row[f"pcy_{B}_candidate_pairs"] = cand
        row[f"pcy_{B}_pass2_MB"] = round((12 * cand + B / 8) / 1e6, 1)
        row[f"pcy_{B}_same_frequent_pairs"] = bool(same)
        assert same, "PCY must find exactly the same frequent pairs as A-Priori"

    i, j = tri_decode(frequent, F)
    pairs = pd.DataFrame({"a_id": freq_ids[i], "b_id": freq_ids[j],
                          "support": all_counts[frequent].astype(np.int64)})
    del all_counts, cand_counts; gc.collect()
    row["status"] = "ok"
    return row, pairs

rows, pairs_by_support = [], {}
for pct in SUPPORTS_PCT:
    buckets = BUCKET_SWEEP if pct == BUCKET_SWEEP_SUPPORT else [MAIN_BUCKETS]
    row, pairs = run_support(pct, buckets)
    rows.append(row)
    if pairs is not None:
        pairs_by_support[pct] = pairs
    print({k: v for k, v in row.items() if not k.endswith("_pct") or k == "support_pct"})

In [ ]:
# 🇫🇷 Contrôle : à 0,05 % on doit retrouver les chiffres d'Alexandre (notebook 02)
check = next(r for r in rows if r["support_pct"] == 0.05)
expected = {"support_baskets": 1608, "frequent_items": 3402,
            "apriori_counted_pairs": 4538892, "frequent_pairs": 5706}
for k, v in expected.items():
    print(f"{k:24s} notebook 02: {v:>10,}   here: {check.get(k, 'n/a'):>10,}   "
          f"{'OK' if check.get(k) == v else 'DIFFERENT'}")

## 4 · Rules and coverage at each support

Same rules as notebook 02: confidence and lift in both directions. A product is **covered** when at least one rule with lift > 1 can recommend it.

In [ ]:
# 🇫🇷 Règles (confiance, lift) et nombre de produits qu'on peut recommander
def rules_from(pairs):
    a, b, c = pairs["a_id"].to_numpy(), pairs["b_id"].to_numpy(), pairs["support"].to_numpy()
    ca, cb = item_counts[a], item_counts[b]
    left = pd.DataFrame({"a_id": a, "b_id": b, "support": c, "confidence": c / ca, "lift": c * N / (ca * cb)})
    right = pd.DataFrame({"a_id": b, "b_id": a, "support": c, "confidence": c / cb, "lift": c * N / (ca * cb)})
    r = pd.concat([left, right], ignore_index=True)
    r.insert(0, "then_buy", r["b_id"].map(NAMES))
    r.insert(0, "if_bought", r["a_id"].map(NAMES))
    return r[["if_bought", "then_buy", "support", "confidence", "lift", "a_id", "b_id"]]

rules_by_support = {}
for row in rows:
    pct = row["support_pct"]
    if pct not in pairs_by_support:
        continue
    r = rules_from(pairs_by_support[pct])
    rules_by_support[pct] = r
    row["rules"] = len(r)
    row["covered_products"] = int(r.loc[r["lift"] > 1, "b_id"].nunique())
    row["coverage_pct_of_catalogue"] = round(100 * row["covered_products"] / len(NAMES), 2)
    print(f"{pct:>6}%  rules {len(r):>8,}   covered products {row['covered_products']:>6,} "
          f"({row['coverage_pct_of_catalogue']}% of the catalogue)")

## 5 · Results

In [ ]:
# 🇫🇷 Tableau récapitulatif
cols = ["support_pct", "support_baskets", "frequent_items", "apriori_triangular_counters",
        "apriori_counted_pairs", f"pcy_{MAIN_BUCKETS}_candidate_pairs", "frequent_pairs",
        "apriori_triangular_MB", "apriori_triples_MB", f"pcy_{MAIN_BUCKETS}_pass2_MB",
        "covered_products", "pass2_s", "status"]
summary = pd.DataFrame(rows)
summary.to_csv(f"{OUT}/summary.csv", index=False)
summary[[c for c in cols if c in summary.columns]]

In [ ]:
# 🇫🇷 Effet du nombre de seaux (au seuil BUCKET_SWEEP_SUPPORT)
r = next(x for x in rows if x["support_pct"] == BUCKET_SWEEP_SUPPORT)
bucket_table = pd.DataFrame([{
    "buckets": B,
    "pass1_bucket_array_MB": round(4 * B / 1e6, 1),
    "frequent_buckets_%": r.get(f"pcy_{B}_frequent_buckets_pct"),
    "candidate_pairs": r.get(f"pcy_{B}_candidate_pairs"),
    "vs_A-Priori_counted_pairs_%": (round(100 * r[f"pcy_{B}_candidate_pairs"] / r["apriori_counted_pairs"], 1)
                                     if r.get("status") == "ok" else None),
    "pass2_MB": r.get(f"pcy_{B}_pass2_MB"),
} for B in BUCKET_SWEEP])
bucket_table.to_csv(f"{OUT}/bucket_sweep.csv", index=False)
bucket_table

In [ ]:
# 🇫🇷 Graphique : compteurs au passage 2 selon le seuil (échelle log)
ok = summary[summary["status"] == "ok"].sort_values("support_pct", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = ok["support_pct"].astype(str) + "%"
axes[0].plot(x, ok["apriori_triangular_counters"], "o-", label="A-Priori, triangular matrix C(F,2)")
axes[0].plot(x, ok["apriori_counted_pairs"], "s-", label="A-Priori, pairs that occur")
axes[0].plot(x, ok[f"pcy_{MAIN_BUCKETS}_candidate_pairs"], "^-", label="PCY candidates")
axes[0].plot(x, ok["frequent_pairs"], "d--", label="frequent pairs (the answer)")
axes[0].set_yscale("log"); axes[0].set_xlabel("support"); axes[0].set_ylabel("pass-2 counters")
axes[0].set_title("Counters held in pass 2"); axes[0].legend(fontsize=8)
axes[1].plot(x, ok["apriori_triangular_MB"], "o-", label="A-Priori, triangular (4 B/counter)")
axes[1].plot(x, ok["apriori_triples_MB"], "s-", label="A-Priori, triples (12 B/pair)")
axes[1].plot(x, ok[f"pcy_{MAIN_BUCKETS}_pass2_MB"], "^-", label="PCY, triples + bitmap")
axes[1].set_xlabel("support"); axes[1].set_ylabel("MB (MMDS memory model)")
axes[1].set_title("Pass-2 memory"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/pcy_vs_apriori.png", dpi=120); plt.show()

In [ ]:
# 🇫🇷 Règles à 0,01 % (plus de produits couverts qu'à 0,05 %) : utilisées par l'agent (04) et l'évaluation (05)
RULES_EXPORT_SUPPORT = 0.01 if 0.01 in rules_by_support else min(rules_by_support)
fname = f"rules_full_support_{str(RULES_EXPORT_SUPPORT).replace('.', 'p')}.csv"
rules_by_support[RULES_EXPORT_SUPPORT].to_csv(f"{OUT}/{fname}", index=False)
size_mb = os.path.getsize(f"{OUT}/{fname}") / 1e6
print("saved", f"{OUT}/{fname}", f"({len(rules_by_support[RULES_EXPORT_SUPPORT]):,} rules, {size_mb:.1f} MB)")
if size_mb > 50:
    print("Warning: over 50 MB, too heavy for GitHub. Keep it on Drive instead.")

In [ ]:
# 🇫🇷 Preuves pour le rapport de la semaine 11
evidence = {
    "baskets": N, "lines": int(len(items)), "pairs_in_baskets": ALL_PAIRS,
    "pass1_apriori_s": round(t_pass1_apriori, 2),
    "pass1_pcy_hash_s": {str(B): round(t, 1) for B, t in t_hash.items()},
    "main_buckets": MAIN_BUCKETS,
    "supports": rows,
    "bucket_sweep": bucket_table.to_dict(orient="records"),
    "rules_file": f"{OUT}/{fname}",
}
with open(f"{OUT}/evidence_week11_pcy.json", "w") as f:
    json.dump(evidence, f, indent=2, default=str)
print(json.dumps({k: v for k, v in evidence.items() if k != "supports"}, indent=2, default=str))

## 6 · What to say at the presentation

Fill in the brackets with the numbers printed above.

> *"In notebook 02, A-Priori at 0.05 % could only recommend 763 products. To cover more, we lowered the support to 0.01 %, which covers [covered] products. At that support, A-Priori's triangular matrix would need [X] counters ([Y] MB). PCY hashes every pair into [16.8 million] buckets in pass 1, and in pass 2 it only counts [Z] candidate pairs, [W] % of what A-Priori counts, and it finds exactly the same frequent pairs."*

> 🇫🇷 *Le point clé : PCY ne change pas la réponse (mêmes paires fréquentes, vérifié), il réduit seulement la mémoire du passage 2. Le prix à payer : un passage 1 plus long, car il faut hacher toutes les paires de tous les paniers. Et avec trop peu de seaux, presque tous les seaux sont fréquents et le filtre ne sert plus à rien (tableau « nombre de seaux »).*

In [ ]:
# 🇫🇷 Télécharger tous les résultats en un zip (à mettre ensuite sur GitHub, dans le dossier out/)
import shutil
zip_path = shutil.make_archive(OUT.rstrip("/").split("/")[-1], "zip", root_dir="out", base_dir=OUT.rstrip("/").split("/")[-1])
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("created", zip_path)